# Modul 4 · Explorative Analyse und Visualisierung
**Data Science 1** · linkit courses

Die Nordwerk GmbH möchte wissen, was in ihren geprüften Werksdaten steckt, bevor jemand ein Modell baut. Du schaust dir Verteilungen und Zusammenhänge an, findest heraus, wo Kennzahlen täuschen, und baust am Ende eine Grafik, mit der die Werksleitung entscheiden kann.

**Lernziele**
1. die Frage vor der Grafik formulieren
2. Verteilungen mit Histogramm, ECDF und Boxplot vergleichen
3. robuste und klassische Kennzahlen begründet wählen
4. Zusammenhänge mit Pearson und Spearman messen
5. das Simpson-Paradox erkennen
6. Grafiken nach Regeln der Wahrnehmung gestalten

**So arbeitest du**
- Das Notebook läuft auf der Kursseite direkt im Browser, ohne Installation. Alternativ öffnest du es in Google Colab und speicherst dort zuerst eine eigene Kopie (*Datei → Kopie in Drive speichern*).
- Führe die Zellen von oben nach unten mit **Shift + Enter** aus.
- Stellen mit `# TODO` füllst du selbst aus, meist mit ein bis acht Zeilen. Darunter prüft ein Selbsttest dein Ergebnis. Ein ❌ ist nur ein Hinweis, dass etwas noch nicht stimmt, das Notebook läuft trotzdem weiter.
- Zellen, die ein Ergebnis aus einem TODO brauchen, melden „⏳ Erst TODO n lösen.“, solange es fehlt.
- Mindestziel sind TODO 1 bis 5, TODO 7 sowie TODO 9 bis 12. TODO 6, TODO 8 und die Bonusaufgaben vertiefen.
- Jede Grafikzelle endet mit `plt.show()`. Die Selbsttests prüfen bei Grafiken die Achsen und die Daten dahinter, nicht das Aussehen. Farben, Größen und Beschriftungen gestaltest du frei, solange die verlangten Eigenschaften stimmen.
- Jeder Abschnitt gehört zu einer Lerneinheit und zum Abschnitt mit derselben Nummer im Skript. Hilfen zu jedem TODO und typische Fehler findest du auf der Kursseite.

In [ ]:
# Hilfsfunktion für die Selbsttests: einfach ausführen, nicht ändern.
import math

def check(name, test):
    """Führt einen Test aus und zeigt ✅ oder ❌. Fehler im Test gelten als „noch nicht gelöst“."""
    try:
        ok = bool(test())
    except Exception as e:
        print(f"❌ {name}: noch nicht gelöst ({type(e).__name__}: {e})")
        return
    print(f"✅ {name}" if ok else f"❌ {name}: Ergebnis stimmt noch nicht")

print("Bereit.")

In [ ]:
# Daten laden: Auf der Kursseite liegen die Dateien schon bereit, in Colab holt diese Zelle sie von GitHub.
import os, urllib.request
DATEN = ["produktion.csv", "stoerungen.csv", "maschinen.csv", "linien.csv"]
QUELLE = "https://raw.githubusercontent.com/juliankunz18/linkit-courses/main/data-science-1/daten/"
os.makedirs("daten", exist_ok=True)
for datei in DATEN:
    if not os.path.exists("daten/" + datei):
        urllib.request.urlretrieve(QUELLE + datei, "daten/" + datei)
print("Daten bereit:", ", ".join(sorted(os.listdir("daten"))))

## 1 · Erst die Frage, dann die Grafik
*Lerneinheit 1 · Skript Abschnitt 1*

Tukey (1977) nennt das Vorgehen **explorative Datenanalyse**: erst schauen, dann modellieren. Jede Grafik beantwortet eine Frage, und fast jede Frage an eine Tabelle gehört zu einem von fünf Typen:

| Fragetyp | Beispiel im Nordwerk | passende Grafik |
|---|---|---|
| Verteilung | Wie lange dauern Störungen? | Histogramm, ECDF, Boxplot |
| Vergleich | Welche Linie hat die höchste Ausschussquote? | Punkte oder Balken je Gruppe, Boxplots nebeneinander |
| Zusammenhang | Steigt der Ausschuss mit der Hallentemperatur? | Streudiagramm |
| Zeitverlauf | Wie entwickelt sich die Quote von Monat zu Monat? | Liniendiagramm |
| Anteil | Welcher Anteil des Stillstands ist mechanisch? | Balken (warum keine Torte, zeigt Abschnitt 6) |

Ab diesem Modul arbeitest du mit den geprüften Dateien. `produktion` bekommt wie in Modul 2 die Spalte `quote`, den Ausschuss geteilt durch die gefertigten Stück.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SCHICHT_TYP = pd.CategoricalDtype(["Früh", "Spät", "Nacht"], ordered=True)
produktion = pd.read_csv("daten/produktion.csv", parse_dates=["datum"], dtype={"schicht": SCHICHT_TYP})
stoerungen = pd.read_csv("daten/stoerungen.csv", parse_dates=["datum"], dtype={"schicht": SCHICHT_TYP})
maschinen = pd.read_csv("daten/maschinen.csv")
linien = pd.read_csv("daten/linien.csv")
produktion["quote"] = produktion["ausschuss_stueck"] / produktion["ist_stueck"]
print(produktion.shape, stoerungen.shape, maschinen.shape, linien.shape)

**Das Objektmodell von matplotlib.** Eine **Figure** ist das ganze Bild, eine **Axes** ein Koordinatensystem darin. `fig, ax = plt.subplots()` legt beides an, danach zeichnest und beschriftest du mit den Methoden von `ax`: `ax.plot`, `ax.set_xlabel`, `ax.set_ylabel`, `ax.set_title`. `plt.show()` zeigt die Figure an. Die monatliche Ausschussquote als Quote der Summen, mit `resample("ME")` aus Modul 2:

In [ ]:
monat = produktion.set_index("datum").resample("ME")[["ausschuss_stueck", "ist_stueck"]].sum()
monat["quote"] = monat["ausschuss_stueck"] / monat["ist_stueck"]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(monat.index, 100 * monat["quote"], marker="o")
ax.set_ylim(0, 3)
ax.set_xlabel("Monat")
ax.set_ylabel("Ausschussquote in %")
ax.set_title("Die Ausschussquote schwankt zwischen 1,9 und 2,5 %")
plt.show()
print(type(fig).__name__, type(ax).__name__, len(fig.axes), len(ax.lines))

**seaborn** baut auf matplotlib auf und erwartet lange Tabellen: eine Zeile je Beobachtung, eine Spalte je Variable. Du nennst Spaltennamen für `x`, `y`, Farbe (`hue=`) und Panels (`col=`).
- **Achsenbezogene** Funktionen wie `sns.histplot`, `sns.ecdfplot`, `sns.scatterplot` zeichnen in eine Axes, die du mit `ax=` übergibst.
- **Figurbezogene** Funktionen wie `sns.relplot`, `sns.displot`, `sns.catplot` legen eine eigene Figure an, mit einem Panel je Gruppe, und geben ein `FacetGrid` zurück. Ein `ax=` ignorieren sie mit einer Warnung.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
zurueck = sns.histplot(data=produktion, x="quote", ax=ax)
plt.show()

g = sns.relplot(data=produktion, x="auslastung", y="quote", col="linie", col_wrap=2,
                height=3, aspect=1.4, s=8, alpha=0.3)
g.set_axis_labels("Auslastung", "Ausschussquote")
plt.show()
print(zurueck is ax, type(g).__name__, g.axes.shape, g.axes[0].get_title())

### TODO 1 · Fragen und Grafiken zuordnen
Ordne jeder Frage einen Fragetyp aus `FRAGETYPEN` und eine passende Grafik aus `GRAFIKEN` zu. Schreib die Namen genau so wie in den Listen.

- **a** Wie lange dauern Störungen, und wie oft dauert eine länger als zwei Stunden?
- **b** Steigt der Ausschuss einer Schicht, wenn es in der Halle wärmer ist?
- **c** Wie hat sich die Ausschussquote seit Oktober 2024 von Monat zu Monat entwickelt?
- **d** Unterscheiden sich die vier Linien in der Ausschussquote ihrer Schichten, und wie stark streuen die Schichten je Linie?
- **e** Welchen Anteil am gesamten Stillstand hat jede Störungskategorie?
- **f** Welche Maschine hat die meisten Störungen gemeldet?

`fragetyp` und `grafik` sind Dictionaries mit den Schlüsseln `"a"` bis `"f"`. Der Selbsttest vergleicht verschlüsselt, damit er die Antworten nicht verrät.

In [ ]:
FRAGETYPEN = ["Verteilung", "Vergleich", "Zusammenhang", "Zeitverlauf", "Anteil"]
GRAFIKEN = ["Histogramm", "ECDF", "Boxplots nebeneinander", "Streudiagramm", "Liniendiagramm", "Balkendiagramm", "Tortendiagramm"]

# TODO 1
fragetyp = {"a": None, "b": None, "c": None, "d": None, "e": None, "f": None}
grafik = {"a": None, "b": None, "c": None, "d": None, "e": None, "f": None}

print(fragetyp)
print(grafik)

In [ ]:
import hashlib

def fingerabdruck(antworten):
    """Kurzer Fingerabdruck einer Liste von Antworten, damit der Test sie nicht verrät."""
    return hashlib.sha256("|".join(antworten).encode("utf-8")).hexdigest()[:10]

check("TODO 1a alles aus den Listen", lambda: sorted(fragetyp) == list("abcdef") and sorted(grafik) == list("abcdef")
      and all(v in FRAGETYPEN for v in fragetyp.values()) and all(v in GRAFIKEN for v in grafik.values()))
check("TODO 1b Fragetypen", lambda: fingerabdruck([fragetyp[k] for k in "abcdef"]) == "62a130bdf8")
check("TODO 1c Grafiken", lambda: fingerabdruck([grafik[k] for k in "abcdef"]) in {"505e2c56b2", "78a6981390", "94318b0848", "650cce0935"})

Für Anteile empfiehlt das Skript Balken statt Torte, weil das Auge Längen auf gemeinsamer Skala genauer vergleicht als Winkel (Abschnitt 6). Bei Frage d geht es um ganze Verteilungen je Gruppe, ein Balken mit dem Mittelwert würde die Streuung verschweigen.

## 2 · Verteilungen: Histogramm, Dichte, ECDF, Boxplot
*Lerneinheit 2 · Skript Abschnitt 2*

Ein **Histogramm** zählt die Werte in Klassen gleicher Breite $h$. Die Faustregel von Freedman und Diaconis (1981) lautet
$$h = 2\,\mathrm{IQR}\; n^{-1/3},$$
mit dem Interquartilsabstand aus Modul 3. `np.histogram_bin_edges(x, bins="fd")` rechnet sie selbst und liefert die Klassengrenzen. Beim Stillstand geht das schief:

In [ ]:
dauer = stoerungen["stillstand_min"]
h_fd_dauer = 2 * (dauer.quantile(0.75) - dauer.quantile(0.25)) * len(dauer) ** (-1 / 3)
kanten = np.histogram_bin_edges(dauer, bins="fd")
anzahl, _ = np.histogram(dauer, bins=kanten)
print("Freedman-Diaconis:", round(h_fd_dauer, 2), "Minuten,", len(kanten) - 1, "Klassen, davon leer:", (anzahl == 0).sum())

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(data=stoerungen, x="stillstand_min", bins=kanten, ax=axes[0])
sns.histplot(data=stoerungen, x="stillstand_min", binwidth=5, ax=axes[1])
axes[0].set_title("Freedman-Diaconis: ein Kamm")
axes[1].set_title("5 Minuten: Auflösung der Daten")
for a in axes:
    a.set_xlim(0, 250)
    a.set_xlabel("Stillstand in Minuten")
plt.show()

Die Stillstände sind auf 5 Minuten gerundet. In eine Klasse von 2,47 Minuten fällt höchstens ein möglicher Wert, also bleibt mehr als jede zweite Klasse leer. Die Regel setzt stetige Daten voraus. Bei gerundeten Daten nimmst du ein Vielfaches der Auflösung.

### TODO 2 · Klassenbreite nach Freedman und Diaconis
Die Ausschussquote ist fein genug abgestuft für die Regel.
1. `h_fd`: die Klassenbreite nach Freedman und Diaconis für `produktion["quote"]`, selbst berechnet mit `quantile` und `len`.
2. `kanten_fd`: die Klassengrenzen, die NumPy mit `bins="fd"` für dieselbe Spalte liefert. Vergleiche die Breite `kanten_fd[1] - kanten_fd[0]` mit `h_fd`.
3. Zeichne in `fig_fd, axes_fd = plt.subplots(1, 2, ...)` zwei Histogramme der Quote nebeneinander: links mit der Klassenbreite 0.01, rechts mit `h_fd`. Beschrifte beide x-Achsen.

In [ ]:
# TODO 2a und 2b
h_fd = None
kanten_fd = None

# TODO 2c: links Klassenbreite 0.01, rechts h_fd
fig_fd, axes_fd = plt.subplots(1, 2, figsize=(10, 4))

plt.show()
print(h_fd, None if kanten_fd is None else len(kanten_fd))

In [ ]:
check("TODO 2a h_fd", lambda: math.isclose(h_fd, 0.0020551, abs_tol=1e-6))
check("TODO 2b kanten_fd", lambda: len(kanten_fd) == 46 and math.isclose(kanten_fd[1] - kanten_fd[0], 0.0020434, abs_tol=1e-6))
check("TODO 2c zwei Histogramme", lambda: len(axes_fd) == 2 and len(axes_fd[0].patches) in (9, 10)
      and 44 <= len(axes_fd[1].patches) <= 46 and all(a.get_xlabel() not in ("", "quote") for a in axes_fd))

NumPy rundet die Zahl der Klassen auf 45 auf und verteilt sie gleichmäßig, deshalb sind seine Klassen etwas schmaler als `h_fd`. Die feine Fassung zeigt die 468 Schichten ohne Ausschuss als eigene Spitze bei 0 und den langen rechten Schwanz. Mit Klassen von 0,01 verschwindet die Spitze in der ersten Klasse.

Die **Kerndichte** ersetzt die Kästchen durch Glocken der Bandbreite $h$, seaborn nimmt $h = s \cdot n^{-1/5}$. Am Rand eines begrenzten Wertebereichs legt sie Fläche, wo keine Werte sein können. Die **ECDF** $\hat F(x)$, der Anteil der Werte bis $x$, kommt ohne Klassenbreite und Bandbreite aus. Quantile liest du direkt ab, und Gruppen legst du übereinander.

In [ ]:
h_kde = dauer.std() * len(dauer) ** (-1 / 5)
print("Bandbreite nach Scott:", round(h_kde, 2), "Minuten")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.kdeplot(data=stoerungen, x="stillstand_min", ax=axes[0])
axes[0].axvline(5, color="grey", linestyle=":")
axes[0].set_xlim(-30, 200)
axes[0].set_title("Kerndichte: Fläche unter 5 Minuten")
sns.ecdfplot(data=stoerungen, x="stillstand_min", hue="schicht", ax=axes[1])
axes[1].set_xlim(0, 200)
axes[1].set_title("ECDF je Schicht")
for a in axes:
    a.set_xlabel("Stillstand in Minuten")
plt.show()
print(stoerungen.groupby("schicht", observed=True)["stillstand_min"].quantile([0.5, 0.9]).unstack())

Früh, spät und nachts dauern Störungen fast gleich lang. Wie sieht es bei den Kategorien aus? Stillstandsdauern sind rechtsschief. Auf einer **Log-Skala** sind gleiche Abstände gleiche Faktoren: Von 5 auf 10 Minuten ist es so weit wie von 100 auf 200.

### TODO 3 · Stillstand je Kategorie als ECDF
1. Zeichne in `fig_ecdf, ax_ecdf` die ECDF der Stillstandsdauer je Kategorie, alle fünf in einer Axes, mit logarithmischer x-Achse. `sns.ecdfplot` kennt dafür `hue=` und `log_scale=`. Beschrifte die x-Achse mit Größe und Einheit.
2. `ueber_60`: der Anteil der Störungen, die länger als 60 Minuten dauern, je Kategorie, absteigend sortiert. An der ECDF ist das $1 - \hat F(60)$.

In [ ]:
# TODO 3a
fig_ecdf, ax_ecdf = plt.subplots(figsize=(8, 4.5))

plt.show()

# TODO 3b
ueber_60 = None

print(ueber_60)

In [ ]:
check("TODO 3a ECDF auf Log-Skala", lambda: ax_ecdf.get_xscale() == "log" and len(ax_ecdf.lines) == 5
      and ax_ecdf.get_xlabel() not in ("", "stillstand_min"))
check("TODO 3b ueber_60", lambda: ueber_60.index[0] == "mechanisch" and ueber_60.is_monotonic_decreasing
      and math.isclose(ueber_60["mechanisch"], 0.2712, abs_tol=1e-4) and math.isclose(ueber_60["elektrisch"], 0.2304, abs_tol=1e-4)
      and math.isclose(ueber_60["Bedienung"], 0.0035, abs_tol=1e-4))

Mechanische und elektrische Störungen dauern viel länger: Mehr als jede vierte mechanische Störung legt die Maschine über eine Stunde still, bei Bedienfehlern fast keine. Der Median der Bedienfehler ist ein Viertel des Medians der mechanischen Störungen. Auch die Streuung unterscheidet sich: Auf der Log-Skala steigt die Kurve der Bedienfehler am steilsten, ihre mittlere Hälfte reicht von 10 bis 15 Minuten, $Q_3$ ist das 1,5-Fache von $Q_1$. Bei elektrischen Störungen reicht sie von 20 bis 60 Minuten, das 3-Fache. Die Kurven sind also keine verschobenen Kopien, Bedienfehler streuen auch relativ am wenigsten.

Ein **Boxplot** zeigt $Q_1$, Median und $Q_3$ als Box. Die **Whisker** reichen bis zum äußersten Wert innerhalb von $Q_1 - 1{,}5\,\mathrm{IQR}$ und $Q_3 + 1{,}5\,\mathrm{IQR}$, die IQR-Regel aus Modul 3. Werte jenseits davon erscheinen als Punkte. `ax.boxplot` von matplotlib nimmt eine Liste von Gruppen und läuft im Browser und in Colab gleich. Die Planerfüllung je Linie aus dem Skript:

In [ ]:
planerfuellung = produktion["ist_stueck"] / produktion["soll_stueck"]
gruppen = [planerfuellung[produktion["linie"] == linie] for linie in [1, 2, 3, 4]]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.boxplot(gruppen)
ax.set_xticks([1, 2, 3, 4], ["Linie 1", "Linie 2", "Linie 3", "Linie 4"])
ax.set_ylabel("Planerfüllung (ist / soll)")
plt.show()

### TODO 4 · Boxplot der Ausschussquote je Linie
1. Zeichne in `fig_box, ax_box` die Boxplots der Ausschussquote je Linie nebeneinander, mit `Linie 1` bis `Linie 4` an der x-Achse und einer Beschriftung der y-Achse.
2. `whisker`: ein DataFrame mit einer Zeile je Linie (Index 1 bis 4) und den Spalten `unten` und `oben`, den Enden der Whisker nach der 1,5-IQR-Regel, sowie `ausreisser`, der Zahl der Schichten über dem oberen Whisker. Schreib dafür eine kleine Funktion, die zu einer Series diese drei Werte liefert, und wende sie mit `groupby` an.

In [ ]:
# TODO 4a
fig_box, ax_box = plt.subplots(figsize=(8, 4.5))

plt.show()

# TODO 4b
whisker = None

print(whisker)

In [ ]:
check("TODO 4a Boxplots", lambda: len(ax_box.get_xticks()) == 4 and ax_box.get_ylabel() != ""
      and [t.get_text() for t in ax_box.get_xticklabels()] == ["Linie 1", "Linie 2", "Linie 3", "Linie 4"])
check("TODO 4b oberer Whisker", lambda: list(whisker.index) == [1, 2, 3, 4]
      and all(math.isclose(a, b, abs_tol=1e-5) for a, b in zip(whisker["oben"], [0.05102, 0.06280, 0.04082, 0.03521])))
check("TODO 4c unten und Ausreißer", lambda: (whisker["unten"] == 0).all() and [int(v) for v in whisker["ausreisser"]] == [22, 18, 22, 12])

In [ ]:
if whisker is None:
    print("⏳ Erst TODO 4 lösen.")
else:
    null = (produktion["quote"] == 0).groupby(produktion["linie"]).sum()
    print(pd.DataFrame({"Q1": produktion.groupby("linie")["quote"].quantile(0.25).round(4),
                        "Median": produktion.groupby("linie")["quote"].median().round(4),
                        "oberer Whisker": whisker["oben"].round(4), "Schichten ohne Ausschuss": null}))

Alle unteren Whisker enden bei 0, weil die untere Grenze negativ ist und Quoten nicht unter 0 fallen. Bei Linie 4 ist sogar $Q_1 = 0$: Mehr als jede vierte Schicht der Endmontage hat keinen Ausschuss. Punkte gibt es nur oben, die Verteilungen sind rechtsschief.

## 3 · Lage und Streuung: Mittelwert, Median, Quantile, MAD
*Lerneinheit 3 · Skript Abschnitt 3*

| Kennzahl | Formel oder pandas | Bruchpunkt |
|---|---|---|
| Mittelwert | $\bar x = \frac{1}{n}\sum_i x_i$, `x.mean()` | $1/n$ |
| Median | mittlerer Wert der sortierten Reihe, `x.median()` | 50 % |
| $p$-Quantil, $P_{90}$ | `x.quantile(0.9)`, linear interpoliert | $\min(p, 1-p)$ |
| Standardabweichung | $s = \sqrt{\frac{1}{n-1}\sum_i (x_i - \bar x)^2}$, `x.std()` | $1/n$ |
| IQR | $Q_3 - Q_1$ | 25 % |
| MAD | $\mathrm{median}_i\,\lvert x_i - \tilde x \rvert$, `(x - x.median()).abs().median()` | 50 % |

Der **Bruchpunkt** ist der kleinste Anteil der Werte, den du beliebig verfälschen musst, damit die Kennzahl beliebig weit wegläuft. Für normalverteilte Daten gilt $\mathrm{IQR} \approx 1{,}349\,\sigma$ und $\mathrm{MAD} \approx 0{,}6745\,\sigma$. Die zehn Störungen aus dem Skript:

In [ ]:
dauer_zehn = pd.Series([5, 10, 15, 15, 20, 25, 30, 40, 60, 180])
std_zehn = dauer_zehn.std()
iqr_zehn = dauer_zehn.quantile(0.75) - dauer_zehn.quantile(0.25)
mad_zehn = (dauer_zehn - dauer_zehn.median()).abs().median()
print("Mittelwert", dauer_zehn.mean(), "| Median", dauer_zehn.median(), "| P90", round(dauer_zehn.quantile(0.9), 2))
print("s", round(std_zehn, 2), "| IQR", iqr_zehn, "| MAD", mad_zehn, "| Variationskoeffizient", round(std_zehn / dauer_zehn.mean(), 2))

ausreisser = dauer_zehn.replace(180, 1800)
print("mit 1800 statt 180: Mittelwert", ausreisser.mean(), "| Median", ausreisser.median())

### TODO 5 · Robuste und klassische Kennzahlen je Kategorie
Erstelle `kennzahlen`: einen DataFrame mit einer Zeile je Störungskategorie und genau den Spalten `mittel`, `median`, `p90`, `std`, `iqr` und `mad` für `stillstand_min`, sortiert nach dem Median aufsteigend. Benannte Aggregation aus Modul 2 nimmt auch eigene Funktionen, etwa `p90=("stillstand_min", lambda x: x.quantile(0.9))`.

In [ ]:
# TODO 5
kennzahlen = None

print(kennzahlen)

In [ ]:
check("TODO 5a Form", lambda: kennzahlen.shape == (5, 6) and list(kennzahlen.columns) == ["mittel", "median", "p90", "std", "iqr", "mad"]
      and kennzahlen["median"].is_monotonic_increasing)
check("TODO 5b mechanisch", lambda: math.isclose(kennzahlen.loc["mechanisch", "mittel"], 53.18, abs_tol=0.01)
      and kennzahlen.loc["mechanisch", ["median", "p90", "iqr", "mad"]].tolist() == [40, 100, 40, 20]
      and math.isclose(kennzahlen.loc["mechanisch", "std"], 45.34, abs_tol=0.01))
check("TODO 5c Bedienung", lambda: kennzahlen.loc["Bedienung", ["median", "p90", "iqr", "mad"]].tolist() == [10, 25, 5, 5]
      and math.isclose(kennzahlen.loc["Bedienung", "mittel"], 14.06, abs_tol=0.01))

In [ ]:
if kennzahlen is None:
    print("⏳ Erst TODO 5 lösen.")
else:
    vergleich_streuung = pd.DataFrame({
        "mittel / median": (kennzahlen["mittel"] / kennzahlen["median"]).round(2),
        "std": kennzahlen["std"].round(1),
        "IQR / 1,349": (kennzahlen["iqr"] / 1.349).round(1),
        "MAD / 0,6745": (kennzahlen["mad"] / 0.6745).round(1),
    })
    print(vergleich_streuung)

In jeder Kategorie liegt der Mittelwert über dem Median, um 16 % bis 41 %, und die Standardabweichung über dem, was IQR und MAD bei einer Normalverteilung erwarten ließen: Der lange Schwanz zieht beide klassischen Kennzahlen nach oben. Für die Einsatzplanung der Instandhaltung taugt $P_{90}$: Bei mechanischen und elektrischen Störungen reichen 100 Minuten in 90 % der Fälle, bei Bedienfehlern 25.

## 4 · Zusammenhänge: Streudiagramm, Pearson, Spearman
*Lerneinheit 4 · Skript Abschnitt 4*

Bei 5.788 Schichten verdecken sich die Punkte eines Streudiagramms. Kleine, durchscheinende Punkte (`alpha`) oder Sechsecke, die Punkte zählen (`hexbin`), zeigen, wo die Masse liegt:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
axes[0].scatter(produktion["hallentemp_c"], produktion["quote"], s=6, alpha=0.15)
axes[1].hexbin(produktion["hallentemp_c"], produktion["quote"], gridsize=25, mincnt=1)
axes[0].set_xlabel("Hallentemperatur in °C")
axes[1].set_xlabel("Hallentemperatur in °C")
axes[0].set_ylabel("Ausschussquote")
plt.show()

**Pearson** misst, wie gut sich die Punkte um eine Gerade legen:
$$r = \frac{\sum_i (x_i - \bar x)(y_i - \bar y)}{\sqrt{\sum_i (x_i - \bar x)^2 \, \sum_i (y_i - \bar y)^2}}.$$
**Spearman** $\rho$ ist Pearson der Ränge, gleiche Werte bekommen den mittleren Rang. $\rho$ misst monotone Zusammenhänge und ist robuster gegen Ausreißer. Die fünf Schichten aus dem Skript, einmal mit einem Ausreißer:

In [ ]:
paare = pd.DataFrame({"temp": [20, 21, 22, 23, 24], "quote_prozent": [1.8, 2.0, 2.5, 2.4, 3.0]})
mit_ausreisser = pd.concat([paare, pd.DataFrame({"temp": [21], "quote_prozent": [6.0]})], ignore_index=True)
print("Pearson:", round(paare["temp"].corr(paare["quote_prozent"]), 4), "| mit Ausreißer:",
      round(mit_ausreisser["temp"].corr(mit_ausreisser["quote_prozent"]), 4))
print("Spearman:", paare.corr(method="spearman").iloc[0, 1], "| mit Ausreißer:",
      round(mit_ausreisser.corr(method="spearman").iloc[0, 1], 4))

### TODO 6 · Pearson und Spearman selbst rechnen
1. `pearson(x, y)`: rechnet $r$ mit der Formel oben, mit NumPy. Die Funktion soll Series und Arrays nehmen, `np.asarray(x, dtype=float)` macht aus beidem ein Array.
2. `spearman(x, y)`: Pearson der Ränge. `pd.Series(x).rank()` vergibt Ränge, bei gleichen Werten den mittleren.
3. `r_temp` und `rho_temp`: beide Koeffizienten für `hallentemp_c` und `quote` über alle Schichten, mit deinen Funktionen.

Der Selbsttest vergleicht mit pandas, auch bei vielen gleichen Werten wie in `personal`.

In [ ]:
# TODO 6
def pearson(x, y):
    return None            # ersetze diese Zeile

def spearman(x, y):
    return None            # ersetze diese Zeile

r_temp = None
rho_temp = None

print(r_temp, rho_temp)

In [ ]:
check("TODO 6a pearson", lambda: math.isclose(pearson(paare["temp"], paare["quote_prozent"]), 0.9482, abs_tol=1e-4)
      and math.isclose(pearson(np.array([1, 2, 3]), np.array([2, 4, 7])), np.corrcoef([1, 2, 3], [2, 4, 7])[0, 1]))
check("TODO 6b spearman", lambda: math.isclose(spearman(paare["temp"], paare["quote_prozent"]), 0.9)
      and math.isclose(spearman(produktion["personal"], produktion["quote"]),
                       produktion[["personal", "quote"]].corr(method="spearman").iloc[0, 1], abs_tol=1e-10))
check("TODO 6c r_temp und rho_temp", lambda: math.isclose(r_temp, 0.2577, abs_tol=1e-4) and math.isclose(rho_temp, 0.2530, abs_tol=1e-4))

Über alle Schichten hängen Hallentemperatur und Ausschussquote schwach zusammen, $r \approx 0{,}26$. Gilt das für jede Linie? Je Linie rechnest du mit `groupby`, hier zuerst für Auslastung und Quote:

In [ ]:
r_auslastung = {}
for linie, t in produktion.groupby("linie"):
    r_auslastung[linie] = t["auslastung"].corr(t["quote"])
print(round(produktion["auslastung"].corr(produktion["quote"]), 3), pd.Series(r_auslastung).round(3).tolist())

Schwach, aber in jeder Linie positiv: Wird eine Schicht voller geplant, steigt der Ausschuss ein wenig. Das brauchst du in Abschnitt 5.

### TODO 7 · Temperatur und Ausschuss je Linie
1. Zeichne mit `sns.relplot` ein Streudiagramm von `hallentemp_c` gegen `quote` mit einem Panel je Linie, zwei Panels pro Zeile, und speichere das Ergebnis als `g_temp`. Kleine, durchscheinende Punkte helfen. Beschrifte die Achsen mit `g_temp.set_axis_labels(...)`.
2. `r_linie`: Pearson $r$ zwischen Hallentemperatur und Ausschussquote je Linie, als Series mit der Linie als Index.

In [ ]:
# TODO 7a
g_temp = None

plt.show()

# TODO 7b
r_linie = None

print(r_linie)

In [ ]:
check("TODO 7a vier Panels", lambda: len(g_temp.axes.flat) == 4 and g_temp.axes.flat[-1].get_xlabel() not in ("", "hallentemp_c"))
check("TODO 7b r_linie", lambda: list(r_linie.index) == [1, 2, 3, 4]
      and all(math.isclose(a, b, abs_tol=1e-4) for a, b in zip(r_linie, [0.2102, 0.2021, -0.0100, -0.0076])))

Nur in den Linien 1 und 2 hängt der Ausschuss mit der Temperatur zusammen, beide zerspanen in Halle 1. In den Linien 3 und 4 ist $r$ praktisch null. Über alle Schichten ist $r$ größer als in jeder Linie, weil Halle 1 wärmer ist und die Linien 1 und 2 ohnehin mehr Ausschuss haben.

Ist Wärme also die Ursache? Warme Schichten liegen im Sommer, wenn auch anderes anders sein kann. **Korrelation ist keine Ursache.** Wie man Ursachen nachweist, zeigen Modul 12 und 13.

Achtung im Browser: `x.corr(y, method="spearman")` für zwei Series braucht das Paket SciPy, das hier fehlt. `df[[a, b]].corr(method="spearman")` und `x.rank().corr(y.rank())` laufen überall.

## 5 · Wenn Kennzahlen täuschen: Anscombe und Simpson
*Lerneinheit 5 · Skript Abschnitt 5*

Anscombe (1973) hat vier Datensätze mit je elf Punkten gebaut, deren Kennzahlen übereinstimmen. Die Werte stehen in der nächsten Zelle, als lange Tabelle mit der Spalte `satz`:

In [ ]:
anscombe = pd.DataFrame({
    "satz": ["I"] * 11 + ["II"] * 11 + ["III"] * 11 + ["IV"] * 11,
    "x": [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5] * 3 + [8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8],
    "y": [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68,
          9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74,
          7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73,
          6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89],
})
print(anscombe.groupby("satz").agg(mittel_x=("x", "mean"), mittel_y=("y", "mean"),
                                   std_x=("x", "std"), std_y=("y", "std")).round(2))

### TODO 8 · Anscombes Quartett
1. `kennzahlen_anscombe`: ein DataFrame mit einer Zeile je Satz und den Spalten `r` (Pearson), `steigung` und `achsenabschnitt` der Ausgleichsgeraden. `np.polyfit(x, y, 1)` liefert Steigung und Achsenabschnitt der Geraden mit der kleinsten Summe quadrierter senkrechter Abstände, in dieser Reihenfolge.
2. `g_anscombe`: vier Panels mit den Punkten je Satz, zwei pro Zeile, und in jedem Panel die Ausgleichsgerade. `sns.lmplot(..., ci=None)` zeichnet Punkte und Gerade in einem Schritt, oder du zeichnest mit `sns.relplot` und ergänzt die Geraden mit `ax.plot` in einer Schleife über `g_anscombe.axes.flat`.

In [ ]:
# TODO 8a
kennzahlen_anscombe = None

# TODO 8b
g_anscombe = None

plt.show()
print(kennzahlen_anscombe)

In [ ]:
check("TODO 8a Kennzahlen", lambda: list(kennzahlen_anscombe.index) == ["I", "II", "III", "IV"]
      and all(abs(v - 0.816) < 0.001 for v in kennzahlen_anscombe["r"])
      and all(abs(v - 0.5) < 0.001 for v in kennzahlen_anscombe["steigung"])
      and all(abs(v - 3.0) < 0.003 for v in kennzahlen_anscombe["achsenabschnitt"]))
check("TODO 8b vier Panels mit Gerade", lambda: len(g_anscombe.axes.flat) == 4 and all(len(a.lines) >= 1 for a in g_anscombe.axes.flat))

Gleiche Kennzahlen, vier ganz verschiedene Bilder: eine Punktwolke um eine Gerade, eine Kurve, eine Gerade mit einem Ausreißer und ein einziger Punkt, der die Steigung allein bestimmt.

Das **Simpson-Paradox** (Simpson 1951): Ein Zusammenhang kehrt sich um, wenn du Gruppen zusammenfasst. Die ausgedachte Tabelle aus dem Skript, zwei Lieferanten für leichte Wellen und schwierige Gehäuse:

In [ ]:
lieferung = pd.DataFrame({
    "lieferant": ["A", "A", "B", "B"],
    "teil": ["Welle", "Gehäuse", "Welle", "Gehäuse"],
    "gefertigt": [100, 300, 300, 100],
    "ausschuss": [2, 24, 9, 10],
})
lieferung["quote"] = lieferung["ausschuss"] / lieferung["gefertigt"]
print(lieferung.pivot(index="teil", columns="lieferant", values="quote"))
gesamt = lieferung.groupby("lieferant")[["ausschuss", "gefertigt"]].sum()
print(gesamt["ausschuss"] / gesamt["gefertigt"])

A ist bei beiden Teilen besser, insgesamt aber schlechter, weil A zu drei Vierteln die schwierigen Gehäuse liefert. Das Teil wirkt auf beides, auf den Lieferanten und auf den Ausschuss: ein **Confounder**.

### TODO 9 · Personal und Ausschuss im Nordwerk
1. `r_gesamt`: Pearson $r$ zwischen `personal` und `quote` über alle Schichten.
2. `r_personal_linie`: dasselbe je Linie, als Series mit der Linie als Index.
3. Zeichne in `fig_simpson, ax_simpson` ein Streudiagramm von Personal und Ausschussquote, farbig nach Linie, mit einer Legende für die vier Linien und beschrifteten Achsen. Da das Personal nur ganze Zahlen annimmt, hilft etwas Transparenz.

In [ ]:
# TODO 9a und 9b
r_gesamt = None
r_personal_linie = None

# TODO 9c
fig_simpson, ax_simpson = plt.subplots(figsize=(8, 5))

plt.show()
print(r_gesamt)
print(r_personal_linie)

In [ ]:
check("TODO 9a r_gesamt", lambda: math.isclose(r_gesamt, -0.2789, abs_tol=1e-4))
check("TODO 9b r_personal_linie", lambda: list(r_personal_linie.index) == [1, 2, 3, 4]
      and all(math.isclose(a, b, abs_tol=1e-4) for a, b in zip(r_personal_linie, [-0.0132, 0.0064, -0.0037, -0.0113])))
check("TODO 9c Streudiagramm nach Linie", lambda: ax_simpson.get_legend() is not None and len(ax_simpson.get_legend().get_texts()) == 4
      and ax_simpson.get_xlabel() not in ("", "personal") and len(ax_simpson.collections) >= 1)

Über alle Schichten scheint mehr Personal den Ausschuss zu senken. Innerhalb jeder Linie ist davon nichts zu sehen. Die Linie bestimmt beides: Die Endmontage braucht viele Hände und erzeugt wenig Ausschuss, die Zerspanung braucht wenige und erzeugt mehr.

Ein Confounder kann einen Unterschied auch **verdecken**. Die Quote der Summen und die mittlere Auslastung je Schicht:

In [ ]:
je_schicht = produktion.groupby("schicht", observed=True).agg(
    ausschuss=("ausschuss_stueck", "sum"), ist=("ist_stueck", "sum"), auslastung=("auslastung", "mean"))
je_schicht["quote"] = je_schicht["ausschuss"] / je_schicht["ist"]
print(je_schicht[["quote", "auslastung"]].round(4))

Die Nacht liegt nur wenig über der Frühschicht. Nachts wird aber geringer ausgelastet, und mit der Auslastung steigt der Ausschuss (Abschnitt 4). Vergleiche deshalb bei gleicher Auslastung.

### TODO 10 · Nacht gegen Früh bei gleicher Auslastung
1. `klasse`: die Auslastung jeder Schicht, mit `pd.cut` in die Klassen $[0{,}6; 0{,}7)$, $[0{,}7; 0{,}8)$, $[0{,}8; 0{,}9)$ und $[0{,}9; 1{,}0)$ eingeteilt (`right=False`).
2. `je_klasse`: ein DataFrame mit einer Zeile je Klasse und einer Spalte je Schicht (`Früh`, `Spät`, `Nacht`), darin die **Quote der Summen**. Gruppiere nach `klasse` und `"schicht"` zugleich, summiere Ausschuss und gefertigte Stück, teile und forme mit `unstack()` um. Bei Kategorien gehört `observed=True` dazu.
3. `anzahl_klasse`: die Zahl der Schichten je Klasse und Schicht in derselben Form.
4. `unterschied`: Nacht minus Früh je Klasse, in Prozentpunkten.

In [ ]:
# TODO 10
klasse = None
je_klasse = None
anzahl_klasse = None
unterschied = None

print(je_klasse)
print(unterschied)

In [ ]:
check("TODO 10a klasse", lambda: klasse.nunique() == 4 and str(klasse.iloc[0]) == "[0.8, 0.9)")
check("TODO 10b je_klasse", lambda: je_klasse.shape == (4, 3) and list(je_klasse.columns) == ["Früh", "Spät", "Nacht"]
      and math.isclose(je_klasse["Früh"].iloc[1], 0.017153, abs_tol=1e-6) and math.isclose(je_klasse["Nacht"].iloc[1], 0.020801, abs_tol=1e-6)
      and math.isclose(je_klasse["Früh"].iloc[2], 0.020525, abs_tol=1e-6) and math.isclose(je_klasse["Nacht"].iloc[2], 0.024256, abs_tol=1e-6))
check("TODO 10c anzahl_klasse", lambda: int(anzahl_klasse.to_numpy().sum()) == 5788 and int(anzahl_klasse["Nacht"].iloc[3]) == 8)
check("TODO 10d unterschied", lambda: (unterschied > 0).all() and math.isclose(unterschied.iloc[1], 0.3649, abs_tol=1e-4)
      and math.isclose(unterschied.iloc[2], 0.3731, abs_tol=1e-4))

In [ ]:
if anzahl_klasse is None or unterschied is None:
    print("⏳ Erst TODO 10 lösen.")
else:
    print("Schichten je Zelle:")
    print(anzahl_klasse)
    print("\nNacht minus Früh insgesamt:", round(100 * (je_schicht.loc["Nacht", "quote"] - je_schicht.loc["Früh", "quote"]), 2), "Prozentpunkte")

Insgesamt liegt die Nacht 0,12 Prozentpunkte über der Frühschicht, bei gleicher Auslastung 0,36 und 0,37, in den beiden Klassen mit genug Schichten beider Sorten. Die Zellen mit 13, 2 oder 8 Schichten taugen nicht für eine Aussage.

## 6 · Wahrnehmung: Position vor Fläche vor Farbe, Achsen, Log-Skala
*Lerneinheit 6 · Skript Abschnitt 6*

- Nach Cleveland und McGill (1984) vergleicht das Auge Positionen auf einer gemeinsamen Skala am genauesten, dann Positionen auf verschobenen Skalen, dann Längen, Richtungen und Winkel, dann Flächen, Volumen und Krümmung, zuletzt Schattierung und Farbsättigung. Balken auf einer gemeinsamen Grundlinie zählen zur ersten Stufe, gestapelte Segmente nur als Längen. Deshalb Balken oder Punkte statt Torte.
- **Balken beginnen bei null**, denn die Länge trägt die Information. Linien dürfen einen Ausschnitt zeigen.
- **Keine zweite y-Achse**: Beide Skalen sind frei gewählt, jede Kreuzung ist ein Artefakt. Besser zwei Panels nebeneinander oder übereinander.
- **Log-Skala** für multiplikative und schiefe Größen, mit „Log-Skala“ an der Achse. Sie kennt keine Null.
- **Farben** für Kategorien nach Okabe und Ito (2002), keine Regenbogenskala. Für geordnete Zahlen eine sequenzielle Skala wie `viridis`.
- **Datentinte mit Augenmaß** (Tufte 1983): Schmuck weglassen, Beschriftung und Einheit bleiben.

Die nächste Zelle setzt die Farben nach Okabe und Ito für alle folgenden seaborn-Grafiken und legt eine kleine Hilfe `komma(ax)` an, die eine Achse mit Dezimalkomma beschriftet. Dann folgt eine Grafik aus einem Werksbericht, absichtlich irreführend gebaut.

In [ ]:
from matplotlib.ticker import FuncFormatter

OKABE_ITO = ["#E69F00", "#56B4E9", "#009E73", "#F0E442", "#0072B2", "#D55E00", "#CC79A7", "#000000"]
sns.set_palette(OKABE_ITO)

def komma(ax, achse="y", stellen=1):
    """Beschriftet eine Achse mit Dezimalkomma, etwa 2,5 statt 2.5."""
    formatierer = FuncFormatter(lambda wert, pos: f"{wert:.{stellen}f}".replace(".", ","))
    (ax.yaxis if achse == "y" else ax.xaxis).set_major_formatter(formatierer)

# So nicht: abgeschnittene Balkenachse und eine zweite y-Achse
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(je_schicht.index.astype(str), 100 * je_schicht["quote"], color="#D55E00")
ax.set_ylim(2.0, 2.25)
ax.set_ylabel("Ausschussquote in %")
ax_rechts = ax.twinx()
ax_rechts.plot(je_schicht.index.astype(str), 100 * je_schicht["auslastung"], color="black", marker="o")
ax_rechts.set_ylabel("Auslastung in %")
ax.set_title("Ausschuss und Auslastung nach Schicht")
plt.show()

Der Balken der Nacht wirkt 2,6-mal so lang wie der der Frühschicht, die Quote liegt aber nur 6 % höher. Und wo die schwarze Linie die Balken schneidet, hängt allein von den zwei frei gewählten Skalen ab.

### TODO 11 · Eine irreführende Grafik reparieren
Zeichne dieselben Zahlen aus `je_schicht` neu, in `fig_neu, (ax_quote, ax_auslastung) = plt.subplots(1, 2, ...)`:
- links die Quote der Summen je Schicht als Balken, rechts die mittlere Auslastung je Schicht als Balken,
- beide y-Achsen beginnen bei null, beide sind mit Größe und Einheit beschriftet,
- die Schichten in der Reihenfolge Früh, Spät, Nacht.

Werte an den Balken (`ax.bar_label`) und das Dezimalkomma mit `komma` sind freiwillig.

In [ ]:
# TODO 11
fig_neu, (ax_quote, ax_auslastung) = plt.subplots(1, 2, figsize=(10, 4))

plt.show()

In [ ]:
def balkenhoehen(ax):
    return np.array([p.get_height() for p in ax.patches])

check("TODO 11a Achsen ab null", lambda: ax_quote.get_ylim()[0] == 0 and ax_auslastung.get_ylim()[0] == 0
      and ax_quote.get_ylabel() != "" and ax_auslastung.get_ylabel() != "")
check("TODO 11b Balken mit den richtigen Werten", lambda: len(ax_quote.patches) == 3 and len(ax_auslastung.patches) == 3
      and np.allclose(balkenhoehen(ax_quote) / balkenhoehen(ax_quote)[0], je_schicht["quote"] / je_schicht["quote"].iloc[0])
      and np.allclose(balkenhoehen(ax_auslastung) / balkenhoehen(ax_auslastung)[0], je_schicht["auslastung"] / je_schicht["auslastung"].iloc[0])
      and [t.get_text() for t in ax_quote.get_xticklabels()] == ["Früh", "Spät", "Nacht"])
check("TODO 11c zwei Panels statt zwei Achsen", lambda: len(fig_neu.axes) == 2 and len(ax_quote.patches) > 0
      and len(ax_auslastung.patches) > 0 and ax_quote.get_position().bounds != ax_auslastung.get_position().bounds)

## 7 · Eine Grafik, die eine Entscheidung trägt
*Lerneinheit 7 · Skript Abschnitt 7*

**Checkliste**
1. Welche Frage beantwortet die Grafik, und passt der Grafiktyp zum Fragetyp?
2. Nennt der Titel den Befund in einem Satz mit Verb?
3. Tragen die Achsen Größe und Einheit, beginnen Balken bei null, steht „Log-Skala“ dran, wo eine ist?
4. Sind Kategorien sinnvoll geordnet?
5. Ist direkt beschriftet, wo es geht, statt mit Legende?
6. Sind die Farben unterscheidbar, auch bei Farbsehschwäche, und ohne Regenbogen?
7. Ist der Vergleich, auf den es ankommt, eine Position auf gemeinsamer Skala?
8. Steht die Zahl der Beobachtungen dabei, wo Gruppen klein sind?

Das Beispiel aus dem Skript: sortiert, direkt beschriftet, das Wichtige hervorgehoben, der Befund im Titel.

In [ ]:
stunden = stoerungen.groupby("maschine")["stillstand_min"].sum().div(60).sort_values()
farben = ["#0072B2" if m == stunden.index[-1] else "#C8C8C8" for m in stunden.index]

fig, ax = plt.subplots(figsize=(8, 6))
balken = ax.barh(stunden.index, stunden, color=farben)
ax.bar_label(balken, fmt="%.0f", padding=3)
ax.set_xlabel("Stillstand in Stunden, Oktober 2024 bis September 2026")
ax.set_title(f"{stunden.index[-1]} steht mit {stunden.iloc[-1]:.0f} Stunden am längsten still")
plt.show()

### TODO 12 · Die Abschlussgrafik
Die Werksleitung fragt: Ist die Nachtschicht schlechter, wenn man fair vergleicht? Zeichne aus `je_klasse` (TODO 10) in `fig_abschluss, ax_abschluss` die Grafik, die diese Frage beantwortet:
- je eine Linie mit Punkten für Früh und Nacht über den vier Auslastungsklassen, die Quote in Prozent (die x-Positionen dürfen 0 bis 3 sein, mit den Klassen als Beschriftung),
- die beiden Linien direkt beschriftet mit `ax.text`, ohne Legende,
- beide Achsen mit Größe und Einheit,
- ein Titel, der den Befund als Satz mit Verb nennt.

Freiwillig: das Dezimalkomma mit `komma(ax_abschluss)`, die Zahl der Schichten je Punkt oder ein Hinweis auf die kleine Zelle.

In [ ]:
# TODO 12
fig_abschluss, ax_abschluss = plt.subplots(figsize=(8, 5))

plt.show()

In [ ]:
VERBEN = {"ist", "sind", "liegt", "liegen", "steigt", "steigen", "sinkt", "sinken", "fällt", "fallen", "bleibt", "bleiben",
          "hat", "haben", "wächst", "wachsen", "übersteigt", "verdoppelt", "erhöht", "zeigt", "zeigen", "macht", "entsteht",
          "produziert", "fertigt", "verursacht", "kostet"}

def titeltext(ax):
    return " ".join(t for t in [ax.get_title(loc="left"), ax.get_title(), ax.get_title(loc="right")] if t)

def zeigt_werte(ax, werte):
    werte = np.asarray(werte, dtype=float)
    return any(len(l.get_ydata()) == len(werte) and (np.allclose(np.asarray(l.get_ydata(), dtype=float), werte)
               or np.allclose(np.asarray(l.get_ydata(), dtype=float), 100 * werte)) for l in ax.lines)

check("TODO 12a Linien für Früh und Nacht", lambda: zeigt_werte(ax_abschluss, je_klasse["Früh"]) and zeigt_werte(ax_abschluss, je_klasse["Nacht"]))
check("TODO 12b direkt beschriftet", lambda: ax_abschluss.get_legend() is None and len(ax_abschluss.texts) >= 2
      and ax_abschluss.get_xlabel() != "" and ax_abschluss.get_ylabel() != "")
check("TODO 12c Titel mit Befund", lambda: len(titeltext(ax_abschluss).split()) >= 4
      and any(w.strip(".,:;!?").lower() in VERBEN for w in titeltext(ax_abschluss).split()))

Der Test prüft nur, ob ein Verb im Titel steht. Ob der Satz stimmt, prüfst du selbst: Trägt die Grafik genau diese Aussage, und sieht man sie auf den ersten Blick?

## Bonus

### Bonus B1 · Kleine Vielfache im Zeitverlauf
1. `monat_linie`: eine lange Tabelle mit den Spalten `linie`, `datum` und `quote`, der Quote der Summen je Linie und Monat. Ein Weg: `produktion.set_index("datum").groupby("linie")[["ausschuss_stueck", "ist_stueck"]].resample("ME").sum()`, dann teilen und mit `reset_index()` zurück in Spalten.
2. `g_monat`: ein Liniendiagramm der Quote über die Monate mit einem Panel je Linie (`sns.relplot(..., kind="line", col=...)`), gemeinsame y-Achse.

In [ ]:
# Bonus B1
monat_linie = None
g_monat = None

plt.show()
print(monat_linie)

In [ ]:
check("B1 monat_linie", lambda: len(monat_linie) == 96 and {"linie", "datum", "quote"} <= set(monat_linie.columns)
      and math.isclose(monat_linie.loc[(monat_linie["linie"] == 2) & (monat_linie["datum"] == "2026-06-30"), "quote"].iloc[0], 0.03786, abs_tol=1e-5))
check("B1 g_monat", lambda: len(g_monat.axes.flat) == 4)

Nur die Linien 1 und 2 haben im Sommer mehr Ausschuss, Linie 2 im Juni 2026 fast 3,8 %. Das passt zur Temperatur aus TODO 7.

### Bonus B2 · Die Kerndichte von Hand
Die Kerndichte mit Gaußkern und Bandbreite $h$ ist $\hat f(x) = \frac{1}{n\,h}\sum_i \varphi\big((x - x_i)/h\big)$ mit $\varphi(u) = e^{-u^2/2}/\sqrt{2\pi}$. Die Fläche links einer Grenze $g$ ist $\frac{1}{n}\sum_i \Phi\big((g - x_i)/h\big)$, mit der Verteilungsfunktion der Standardnormalverteilung $\Phi(z) = \tfrac{1}{2}\big(1 + \mathrm{erf}(z/\sqrt{2})\big)$ und `math.erf`.
1. `dichte(stellen)`: $\hat f$ für die Stillstandsdauer an den Stellen eines Arrays, mit `h_kde` aus Abschnitt 2. Tipp: Es gibt nur 77 verschiedene Dauern, rechne mit `dauer.value_counts()` und gewichte mit den Häufigkeiten.
2. `anteil_unter_5` und `anteil_unter_0`: die Fläche unter $\hat f$ links von 5 und links von 0 Minuten.

In [ ]:
# Bonus B2
def dichte(stellen):
    return None            # ersetze diese Zeile

anteil_unter_5 = None
anteil_unter_0 = None

print(anteil_unter_5, anteil_unter_0)

In [ ]:
gitter = np.linspace(-50, 550, 6001)
check("B2 dichte", lambda: math.isclose(dichte(np.array([25.0]))[0], 0.019897, abs_tol=1e-6)
      and math.isclose(float(np.sum(dichte(gitter)) * (gitter[1] - gitter[0])), 1, abs_tol=1e-3))
check("B2 Fläche am Rand", lambda: math.isclose(anteil_unter_5, 0.0491, abs_tol=1e-4) and math.isclose(anteil_unter_0, 0.0130, abs_tol=1e-4))

Knapp 5 % der Fläche liegen unter 5 Minuten, wo keine Störung erfasst wird, und gut 1 % bei negativen Dauern. Das ist der Randfehler der Kerndichte aus dem Skript, Abschnitt 2.2.

## Zum Nachdenken

1. Die Regel von Freedman und Diaconis lieferte beim Stillstand einen Kamm, bei der Ausschussquote ein brauchbares Histogramm. Woran erkennst du vorher, ob eine Spalte gerundet ist?
2. Für die Einsatzplanung der Instandhaltung: Würdest du je Kategorie mit dem Mittelwert, dem Median oder $P_{90}$ planen? Und für das Jahresbudget der Stillstandskosten?
3. Über alle Schichten ist $r$ zwischen Temperatur und Ausschuss größer als in jeder einzelnen Linie. Wie kann eine Korrelation über alle größer sein als in jeder Gruppe?
4. In TODO 10 hast du nach der Auslastung geschichtet. Welche andere Spalte könnte den Vergleich von Nacht und Früh noch verzerren, und wie würdest du das prüfen?
5. Der Selbsttest zu TODO 12 akzeptiert den Titel „Die Nacht ist schlecht“. Was fehlt diesem Titel?

<details><summary>Hinweise zu den Antworten</summary>

1. Zähle die verschiedenen Werte und schau auf die kleinsten Abstände zwischen ihnen, etwa mit `np.diff(np.sort(x.unique()))`. Beim Stillstand sind alle Abstände Vielfache von 5.
2. Für die Planung einzelner Einsätze $P_{90}$, damit die Zeit fast immer reicht. Für das Budget zählt die Summe, also der Mittelwert mal die Zahl der Störungen. Der Median unterschätzt beides.
3. Die Linien unterscheiden sich in Temperatur und Ausschuss zugleich: Halle 1 ist wärmer, und ihre Linien haben mehr Ausschuss. Dieser Unterschied zwischen den Gruppen kommt zur Korrelation innerhalb der Gruppen hinzu.
4. Die Linie zum Beispiel, wenn nachts andere Linien liefen als früh. Hier hat jede Linie gleich viele Nachtschichten, zusätzlich kannst du je Linie und Klasse vergleichen. Auch Monat oder Wochentag kommen in Frage, etwa wegen der Samstagsschichten, die es nur früh gibt.
5. Der Bezug auf den fairen Vergleich und eine Größe: Schlecht worin, und im Vergleich zu wem? „Nachts liegt der Ausschuss bei gleicher Auslastung höher“ sagt, was die Grafik zeigt.
</details>

## Wie es weitergeht

Du hast gesehen, dass die Nacht bei gleicher Auslastung mehr Ausschuss hat als die Frühschicht, und dass nur zwei Linien auf Wärme reagieren. Aber ist ein Unterschied von gut einem Drittel Prozentpunkt echt, oder hätte er auch zufällig entstehen können? **Modul 5 „Zufall simulieren: Wahrscheinlichkeit, Bootstrap, Permutation“** beantwortet das mit Simulationen: Du lässt den Zufall selbst Daten erzeugen und siehst, wie stark Kennzahlen schwanken, auch ohne Formeln.